# 인사 데이터 JSONL 변환

### 라이브러리 선언

In [1]:
# 데이터 처리 라이브러리
import pandas as pd
# JSON 파일 읽기/쓰기 라이브러리 (파이썬 기본 내장)
import json
# 파일 경로 처리 라이브러리
from pathlib import Path
# 운영체제 환경변수 읽기
import os
# .env 파일의 환경변수를 불러오는 라이브러리
from dotenv import load_dotenv

print(f'pandas 버전: {pd.__version__}')
print('라이브러리 로딩 완료!')

pandas 버전: 3.0.3
라이브러리 로딩 완료!


### 환경 설정 *** 경로 확인 필요

In [2]:
# .env 파일의 설정값을 환경변수로 불러옴
load_dotenv()

# 전처리된 CSV 파일이 있는 폴더 경로
INPUT_DIR  = Path(os.getenv('INPUT_DIR',  '../데이터 전처리 정제/output'))
# 변환된 JSONL 파일을 저장할 폴더 경로
OUTPUT_DIR = Path(os.getenv('OUTPUT_DIR', 'output'))

# 출력 폴더가 없으면 자동으로 생성
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'  입력 디렉토리: {INPUT_DIR}')
print(f'  출력 디렉토리: {OUTPUT_DIR}')

  입력 디렉토리: ..\데이터 전처리 정제\output
  출력 디렉토리: output


# 1. 데이터 로딩

In [3]:
# 입력 폴더에서 .csv 파일 목록을 이름순으로 가져옴
csv_files = sorted(INPUT_DIR.glob('*.csv'))

if not csv_files:
    # CSV 파일이 없으면 안내 메시지 출력 후 중단
    print(f'CSV 파일 없음: {INPUT_DIR}')
    print('전처리 노트북을 먼저 실행해 주세요.')
else:
    # 파일 이름(확장자 제외)을 키로, DataFrame을 값으로 저장하는 딕셔너리
    dfs = {}
    for path in csv_files:
        # CSV 파일을 읽어 DataFrame으로 변환
        # utf-8-sig: 엑셀 호환 BOM 처리 / dtype=object: 모든 컬럼을 문자열로 읽어 타입 오류 방지
        df = pd.read_csv(path, encoding='utf-8-sig', dtype=object)
        # path.stem: 확장자를 뺀 파일 이름 (예: '기본인사정보_정제')
        dfs[path.stem] = df
        print(f'  로딩: {path.name}  ({len(df):,}행 / {len(df.columns)}열)')

    # 다른 파일에서 부서·직급을 채울 때 사용할 기본인사정보 DataFrame
    # 파일이 없으면 빈 DataFrame으로 초기화해 오류 방지
    basic_df = dfs.get('기본인사정보_정제', pd.DataFrame())

    # 사원번호를 키로, 이름·부서·직급을 값으로 저장하는 조회용 딕셔너리
    basic_lookup = {}
    if not basic_df.empty:
        # to_dict('records'): DataFrame의 각 행을 {'컬럼명': 값} 형태의 딕셔너리로 변환
        for row in basic_df.to_dict('records'):
            emp_id = str(row.get('사원번호', '')).strip()
            basic_lookup[emp_id] = {
                '이름': str(row.get('이름', '')),
                '부서': str(row.get('부서', '')),
                '직급': str(row.get('직급', '')),
            }

    print(f'\n로딩 완료! 총 {len(dfs)}개 파일')
    print(f'기본인사정보 조회 딕셔너리: {len(basic_lookup):,}건')

  로딩: 급여정보_정제.csv  (2,000행 / 7열)
  로딩: 기본인사정보_정제.csv  (2,000행 / 30열)
  로딩: 역량성과_정제.csv  (2,000행 / 13열)
  로딩: 통합인사정보_정제.csv  (2,000행 / 48열)

로딩 완료! 총 4개 파일
기본인사정보 조회 딕셔너리: 2,000건


# 2. 변환 함수 정의

In [4]:
# 빈 값으로 처리할 문자열 목록 (이 값들은 필드에서 제외)
MISSING_VALUES = {'', '미입력', 'nan', 'NaN', 'None', 'none'}

# embedding_text에 포함할 추가 필드 목록
# 이름·부서·직급은 basic_lookup에서 별도 처리하므로 여기에 포함하지 않음
EMBEDDING_FIELDS = [
    '팀', '직책', '입사일', '이메일',
    '성과점수', '자격증', '포상이력',
    '연봉', '급여은행',
]


def clean(val):
    # 값을 문자열로 변환하고 앞뒤 공백 제거
    cleaned = str(val).strip()
    # MISSING_VALUES에 해당하면 빈 문자열 반환, 아니면 그대로 반환
    return cleaned if cleaned not in MISSING_VALUES else ''


def build_embedding_text(row, info):
    # 이름·부서·직급을 가장 앞에 배치 (basic_lookup에 없으면 row에서 직접 읽음)
    parts = [
        info.get('이름', clean(row.get('이름', ''))),
        info.get('부서', clean(row.get('부서', ''))),
        info.get('직급', clean(row.get('직급', ''))),
    ]
    # EMBEDDING_FIELDS에 있는 나머지 필드를 순서대로 추가
    for field in EMBEDDING_FIELDS:
        val = clean(str(row.get(field, '')))
        if val:
            parts.append(val)

    # 중복 제거: 이미 추가된 값은 건너뜀
    seen = []
    for p in parts:
        if p and p not in seen:
            seen.append(p)
    # 공백으로 이어붙여 하나의 문장으로 만듦
    return ' '.join(seen)


def to_record(row):
    # 사원번호를 키로 basic_lookup에서 부서·직급 조회
    emp_id = str(row.get('사원번호', '')).strip()
    # basic_lookup에 없으면 빈 딕셔너리 반환 → info.get()이 기본값을 씀
    info   = basic_lookup.get(emp_id, {})

    # OpenSearch에 저장될 JSONL 한 줄의 구조
    return {
        'employee_id':      emp_id,
        'department':       info.get('부서', clean(row.get('부서', ''))),
        'position':         info.get('직급', clean(row.get('직급', ''))),
        'embedding_text':   build_embedding_text(row, info),
        # 임베딩 벡터는 4단계에서 채워짐 — 지금은 빈 리스트로 초기화
        'embedding_vector': [],
    }


print('변환 함수 정의 완료!')

변환 함수 정의 완료!


# 3. JSONL 변환 및 저장

In [5]:
# 저장된 파일 경로와 건수를 기록할 리스트 (4단계 확인용)
saved_files = []

# dfs 딕셔너리를 순회: source_name = 파일 이름, df = DataFrame
for source_name, df in dfs.items():
    # 출력 파일 경로 (예: output/기본인사정보_정제.jsonl)
    out_path = OUTPUT_DIR / f'{source_name}.jsonl'

    # DataFrame의 모든 행을 딕셔너리 리스트로 변환
    rows = df.to_dict('records')

    # 파일을 쓰기 모드로 열기 (같은 이름 파일이 있으면 덮어씀)
    with open(out_path, 'w', encoding='utf-8') as f:
        for row in rows:
            # 각 행을 JSONL 레코드로 변환
            record = to_record(row)
            # ensure_ascii=False: 한글을 그대로 저장 (유니코드 이스케이프 방지)
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

    saved_files.append((out_path, len(rows)))
    print(f'  저장: {out_path.name}  ({len(rows):,}건)')

  저장: 급여정보_정제.jsonl  (2,000건)
  저장: 기본인사정보_정제.jsonl  (2,000건)
  저장: 역량성과_정제.jsonl  (2,000건)


  저장: 통합인사정보_정제.jsonl  (2,000건)


# 4. 저장 결과 확인

In [6]:
for out_path, _ in saved_files:
    # 저장된 JSONL 파일을 읽기 모드로 열기
    with open(out_path, 'r', encoding='utf-8') as f:
        # readline(): 파일의 첫 번째 줄(=첫 번째 레코드)만 읽음
        # json.loads(): JSON 문자열을 파이썬 딕셔너리로 변환
        sample = json.loads(f.readline())
    print(f'\n[{out_path.name}] 첫 번째 레코드:')
    # indent=2: 들여쓰기 2칸으로 보기 좋게 출력
    print(json.dumps(sample, ensure_ascii=False, indent=2))
    print()


[급여정보_정제.jsonl] 첫 번째 레코드:
{
  "employee_id": "EMP0001",
  "department": "마케팅부",
  "position": "사원",
  "embedding_text": "지준서 마케팅부 사원 32000000 우리은행",
  "embedding_vector": []
}


[기본인사정보_정제.jsonl] 첫 번째 레코드:
{
  "employee_id": "EMP0001",
  "department": "마케팅부",
  "position": "사원",
  "embedding_text": "지준서 마케팅부 사원 브랜드팀 팀원 2024-05-02 jijunseo16@nate.com",
  "embedding_vector": []
}


[역량성과_정제.jsonl] 첫 번째 레코드:
{
  "employee_id": "EMP0001",
  "department": "마케팅부",
  "position": "사원",
  "embedding_text": "지준서 마케팅부 사원 62",
  "embedding_vector": []
}


[통합인사정보_정제.jsonl] 첫 번째 레코드:
{
  "employee_id": "EMP0001",
  "department": "마케팅부",
  "position": "사원",
  "embedding_text": "지준서 마케팅부 사원 브랜드팀 팀원 2024-05-02 jijunseo16@nate.com 62 32000000 우리은행",
  "embedding_vector": []
}

